# 실습 E. 정리 규칙을 코드 전체에 적용하기 (약 35분)

> **조건부 규칙을 적용하는 곳에서 조건을 누락하면 잘못된 결과가 나온다.**

**왜 정리하나요.** 오래된 Java 코드에는 지금 보면 번거로운 옛 방식이 많습니다. 목록이 비었는지 `size() == 0`으로 묻고, 목록을 돌 때 인덱스 `i`를 쓰고, 한 번만 쓰는 임시 변수를 따로 둡니다. 이런 곳을 같은 뜻의 더 읽기 쉬운 코드로 바꾸는 것이 코드 정리(리팩터링)입니다. 같은 모양이 코드 곳곳에 있으므로 보통은 "이 규칙을 코드 전체에 적용해 줘"라고 도구나 AI에게 맡깁니다. "같은 뜻으로 바꾸기"라서 안전해 보이기 때문입니다.

**하지만** 대부분의 정리 규칙은 **어떤 조건에서만** 같은 뜻입니다. 코드 전체에 일괄 적용하는 도구나 AI는 그 조건을 자리마다 확인하지 않을 수 있습니다.

**이 실습의 규칙 세 개.** 예는 HomeHub 코드에서 실제로 바뀌는 곳 하나씩입니다.

| 규칙 | 바꾸기 전 | 바꾼 뒤 |
|---|---|---|
| ① 비었는지 묻기<br>`size() == 0` → `isEmpty()` | `return running.size() == 0;`<br>(`PowerBudget.isIdle`) | `return running.isEmpty();` |
| ② 인덱스 반복 → for-each | `for (int i = 0; i < derived.size(); i++) {`<br>`    dispatcher.submit(derived.get(i));`<br>`}`<br>(`HomeHub.flushDerived`) | `for (Event d : derived) {`<br>`    dispatcher.submit(d);`<br>`}` |
| ③ 변수 인라인<br>한 번만 쓰는 지역 변수를 그 식으로 바꿈 | `String line = tag + ...;`<br>`entries.add(line);`<br>(`AuditLog.record`) | `entries.add(tag + ...);` |


In [ ]:
import lab   # 실습 도우미: lab.run(...)이 make 명령을 실행하고 결과 보고서를 아래에 보여 줍니다

## E-1. 규칙마다 조건 적기 (10분, 직접 채우기)

위 세 규칙이 **언제** 같은 뜻이 되는지 생각해 봅니다. 이 셀을 더블클릭해서 둘째 칸을 채우세요. 셋째 칸은 E-2가 끝난 뒤 채웁니다.

| 규칙 | 같은 뜻이 되는 조건 | 이 코드에서 조건이 맞는가 |
|---|---|---|
| ① `list.size() == 0` → `list.isEmpty()` | | |
| ② 인덱스 반복 `for (int i = 0; i < list.size(); i++) { ... list.get(i) ... }` → `for (T x : list) { ... x ... }` | | |
| ③ 변수 인라인: `int n = e; ... n ...` → `... e ...` | | |


## E-2. 적용하고 확인하기 (15분)

세 규칙을 레거시 코드 **전체에, 적용할 수 있는 곳마다** 적용한 결과가 `scenarios/rw/<규칙>/`에 있습니다. "AI에게 이 정리 규칙을 코드 전체에 적용하라고 시킨 결과"에 해당합니다. 세 가지로 확인합니다.

1. 기존 테스트 38개
2. 기록해 둔 동작 3개: 레거시에 입력 파일 3개(`legacy/workloads/`, 가전이 보낸 메시지를 모은 것)를 넣고, 메시지 한 줄을 처리할 때마다 허브 상태를 한 줄씩 적어 둔 기록(`golden/`)입니다. 정리한 코드에 같은 입력을 넣고 이 기록과 비교합니다
3. 무작위 입력 파일 200개로 레거시와 전체 동작 비교. 다르면 같은 차이가 나는 가장 짧은 입력을 남깁니다

보고서에 나오는 기록 이름 세 개는 이런 상황입니다.

| 기록 | 입력 파일 | 어떤 상황인가 |
|---|---|---|
| `normal_day` | `legacy/workloads/normal_day.txt` (30줄) | 평소 하루. 대기 이벤트가 몇 개 넘지 않음 |
| `burst` | `legacy/workloads/burst.txt` (23줄) | 저녁 피크. 여러 기기가 한꺼번에 보고해 대기열이 가득 참(거부·밀어내기가 일어남) |
| `fridge_firmware` | `legacy/workloads/fridge_firmware.txt` (22줄) | 냉장고 2대. 한 대는 문이 열려 온도가 오르고, 한 대는 화씨로 보고함 |


In [1]:
lab.run("rewrite-check", R="IsEmpty")

$ make rewrite-check R=IsEmpty
tests: 38, passed: 38, failed: 0
기록 normal_day       같음
기록 burst            같음
기록 fridge_firmware  같음
200 random workloads (40 lines each): identical traces


In [2]:
lab.run("rewrite-check", R="ForEach")

$ make rewrite-check R=ForEach
tests: 38, passed: 38, failed: 0
기록 normal_day       같음
기록 burst            같음
기록 fridge_firmware  같음
200 random workloads (40 lines each): identical traces


In [3]:
lab.run("rewrite-check", R="InlineVariable")

$ make rewrite-check R=InlineVariable
tests: 38, passed: 38, failed: 0
기록 normal_day       같음
기록 burst            다름 (10줄)
기록 fridge_firmware  같음
MISMATCH (run 0, seed 7): minimal workload with 9 lines written to out/hubdiff-InlineVariable.txt


**변수 인라인만 동작이 바뀝니다.** 테스트는 38/38 통과하지만, 기록해 둔 동작 `burst`가 다르고 무작위 비교에서도 차이가 나옵니다. 보고서의 "바뀐 코드"에서 세 곳 중 어디가 원인인지 찾아보세요. 찾았으면 아래를 봅니다.

In [4]:
lab.side_by_side([
    (("legacy/src/main/java/com/dxlab/homehub/dispatch/Dispatcher.java", 85, 97),
     ("scenarios/rw/InlineVariable/src/main/java/com/dxlab/homehub/dispatch/Dispatcher.java", 85, 96),
     "레거시는 반복 횟수를 시작할 때 <code>n = queue.size()</code>로 <b>한 번</b> 정해 둠. 인라인하면 반복할 때마다 <code>queue.size()</code>를 <b>다시</b> 읽음"),
], title="Dispatcher.evictOldestNonCritical: 변수 n을 인라인", headers=("레거시", "변수 n을 인라인", "차이"))

**이 메서드가 하는 일.** 대기열이 가득 찼는데 중요 이벤트가 들어오면, 가장 오래된 비중요 이벤트 하나를 빼서 자리를 만듭니다(규칙 R3). 대기열에서 한 개씩 꺼내(`poll`) 다시 뒤에 넣으며(`offer`) 한 바퀴 돌리고, 처음 만난 비중요 이벤트만 다시 넣지 않습니다.

**왜 결과가 달라지나.** 대기열 크기는 반복 중에 바뀝니다. 밀어낼 이벤트를 꺼낼 때 한 번 줄고, 나머지는 꺼냈다 다시 넣으니 그대로입니다. 대기열에 비중요 이벤트 1~8이 들어 있고 중요 이벤트 11이 들어오는 경우를 한 단계씩 보면 이렇습니다.

| `i` | 레거시: `i < n` (n = 8) | 인라인: `i < queue.size()` | 이번에 하는 일 | 인라인 쪽 대기열 |
|---|---|---|---|---|
| 시작 | | | | [1, 2, 3, 4, 5, 6, 7, 8] |
| 0 | 0 < 8, 계속 | 0 < 8, 계속 | 1을 꺼내 밀어낼 이벤트로 둠(다시 넣지 않음) → 크기 7 | [2, 3, 4, 5, 6, 7, 8] |
| 1 | 1 < 8, 계속 | **1 < 7**, 계속 | 2를 꺼내 뒤에 다시 넣음 | [3, 4, 5, 6, 7, 8, 2] |
| 2~6 | 계속 | 계속 | 3, 4, 5, 6, 7을 차례로 꺼내 뒤에 넣음 | [8, 2, 3, 4, 5, 6, 7] |
| 7 | 7 < 8, 계속: 8을 꺼내 뒤에 넣음 → [2, 3, 4, 5, 6, 7, 8] | **7 < 7 거짓: 멈춤** | 인라인 쪽은 8을 돌리지 못함 | [8, 2, 3, 4, 5, 6, 7] |

마지막으로 11을 뒤에 넣으면, 레거시는 **[2, 3, 4, 5, 6, 7, 8, 11]**, 인라인은 **[8, 2, 3, 4, 5, 6, 7, 11]**입니다. 8이 맨 앞으로 와서 남은 이벤트의 순서가 바뀌었습니다(규칙 R3 위반).

변수 인라인이 같은 뜻이 되는 조건은 "변수를 쓰는 동안 그 식의 값이 바뀌지 않는다"입니다. 여기서는 반복 안의 `poll`이 `queue.size()`의 값을 바꾸므로 조건이 깨집니다. 다른 두 곳(`AuditLog.record`, `AuditLog.tail`)은 이 조건이 맞습니다. 테스트는 밀어낸 개수와 빠진 이벤트만 보고 남은 순서는 보지 않아서 통과했습니다.

for-each 규칙에도 조건("반복 중에 목록을 바꾸지 않는다")이 있지만, 이 코드에서는 적용한 8곳 모두 조건이 맞아서 동작이 그대로입니다.

세 규칙을 모두 적용한 결과도 확인해 봅니다.


<details><summary><b>E-1 표의 답 보기</b></summary>

| 규칙 | 같은 뜻이 되는 조건 | 이 코드에서 조건이 맞는가 |
|---|---|---|
| `size() == 0` → `isEmpty()` | 없음(표준 컬렉션이면 늘 같음) | 3곳 모두 맞음 |
| 인덱스 반복 → for-each | 반복 중에 목록을 바꾸지 않음 | 8곳 모두 맞음 |
| 변수 인라인 | 변수를 쓰는 동안 식의 값이 바뀌지 않음(식에 side-effect 도 없음) | `AuditLog` 두 곳은 맞음. **`Dispatcher.evictOldestNonCritical`의 `n = queue.size()`는 깨짐** |
</details>

In [5]:
lab.run("rewrite-check", R="All")

$ make rewrite-check R=All
tests: 38, passed: 38, failed: 0
기록 normal_day       같음
기록 burst            다름 (10줄)
기록 fridge_firmware  같음
MISMATCH (run 0, seed 7): minimal workload with 9 lines written to out/hubdiff-All.txt


**AI에게 직접 시켜 보기.** 아래 셀은 변수 인라인 규칙을 `Dispatcher.java`에 적용하라고 Codex에게 시킵니다. AI가 조건을 확인했습니까? `evictOldestNonCritical`의 `n`을 인라인했습니까?

먼저 쓸 모델과 추론 수준을 고릅니다. 바꾸지 않으면 실습 기본값(`gpt-5.6-luna`, 추론 `none`: 빠르고 토큰이 적음)을 씁니다. 추론 수준을 올리면 결과가 나아질 수 있지만 오래 걸리고 토큰도 많이 씁니다.

In [ ]:
lab.codex_models()          # 이 컴퓨터의 Codex에서 쓸 수 있는 모델과 추론 수준
모델 = "gpt-5.6-luna"        # 바꾸려면 위 목록의 이름으로
추론 = "none"                # none(가장 빠름), low, medium, high, ...

In [6]:
lab.codex("P-5", 규칙="변수 인라인: 한 번만 쓰는 지역 변수를 그 식으로 바꾼다. int n = e; ... n ... → ... e ...", 코드="legacy/src/main/java/com/dxlab/homehub/dispatch/Dispatcher.java",
          model=모델, effort=추론)

Codex에 로그인되어 있지 않습니다. 터미널에서 codex login 을 실행하거나, CODEX_API_KEY 환경 변수를 설정한 뒤 Jupyter를 다시 시작하세요.
Codex를 쓸 수 없으면 lab.prompt('P-5')로 프롬프트를 출력해 사내 AI 도구에 붙여 넣으세요.


## E-3. 규칙이 같은 뜻이라는 것을 Dafny로 증명하기 (10분)

정리 규칙이 "같은 뜻"이라는 것을 Dafny로 이렇게 적습니다.

1. 바꾸기 **전**의 코드와 바꾼 **뒤**의 코드가 하는 일을 각각 Dafny 함수로 적습니다.
2. "두 결과가 같다"를 보조정리(`lemma`)의 사후조건(`ensures`)으로 적고 증명합니다. 이것이 `<규칙>_Same`입니다.
3. 같아지려면 필요한 **조건**은 그 보조정리의 사전조건(`requires`)으로 적습니다. 이것이 E-1에서 적은 "같은 뜻이 되는 조건"입니다.
4. 조건부 규칙은 "조건이 없으면 두 쪽이 다르다"는 구체적인 예도 증명해 둡니다. 이것이 `<규칙>_Counterexample`입니다.

세 규칙이 모두 `rewrite/RewriteRules.dfy` 한 파일에 있습니다. 먼저 검증합니다.

In [7]:
lab.run("rules")

$ make rules
Dafny RewriteRules.dfy                         검증 15개, 오류 0개


B에서처럼 줄마다 색과 설명을 붙였습니다. **주황(사전조건 `requires`)이 규칙의 조건**이고, **청록(사후조건 `ensures`)이 "두 쪽이 같다" 또는 "다르다"**입니다. 흐린 부분은 Dafny가 증명을 찾도록 돕는 계산 과정이라 건너뛰어도 됩니다.

**① `size() == 0` → `isEmpty()`: 조건 없음**

In [8]:
lab.annotate("rewrite/RewriteRules.dfy", [
    (11, 12, "body", "규칙 ①. 규칙마다 <code>module</code> 하나"),
    (13, 13, "body", ""),
    (14, 14, "body", "보조정리(<code>lemma</code>): 증명할 사실 하나. Java의 <code>List</code>는 Dafny의 <code>seq</code>(순서 있는 목록)로 나타냄"),
    (15, 15, "post", "<b>증명할 것</b>: \"길이가 0이다\"(바꾸기 전 <code>size() == 0</code>)와 \"빈 목록이다\"(바꾼 뒤 <code>isEmpty()</code>)는 늘 같은 참/거짓"),
    (16, 17, "body", "본문이 비어 있음: Dafny가 혼자 증명함. <code>requires</code>가 없으니 <b>조건 없이</b> 늘 같음"),
    (18, 18, "body", ""),
], start=11, end=18, title="RewriteRules.dfy: 규칙 ①")

**② 인덱스 반복 → for-each: 조건 "반복하는 동안 루프 body가 목록을 바꾸지 않음"**

In [9]:
lab.annotate("rewrite/RewriteRules.dfy", [
    (20, 26, "body", "규칙 ②. 반복의 루프 body는 \"(지금 원소, 지금 목록) → 루프 body가 끝난 뒤의 목록\" 함수 <code>body</code>로 나타냄. 두 반복이 <b>어떤 원소를 어떤 순서로 방문하는지</b> 비교"),
    (27, 27, "body", ""),
    (28, 34, "func", "<b>바꾸기 전: 인덱스 반복.</b> 매번 <b>지금 목록</b>에서 i번째를 읽고, 루프 body가 바꾼 목록으로 다음 반복을 함. 방문한 원소 목록을 돌려줌. <code>fuel</code>은 반복이 끝난다는 근거"),
    (36, 37, "func", "<b>바꾼 뒤: for-each.</b> 시작할 때의 목록을 처음부터 끝까지 방문"),
    (39, 48, "proof", "증명 도우미: i번째부터 방문한 것이 목록의 i번째 이후와 같다는 것을 하나씩 늘려 가며 증명"),
    (50, 50, "body", "<b>같음 보조정리</b>"),
    (51, 51, "pre", "<b>조건</b>: 루프 body가 목록을 바꾸지 않음. 어떤 원소, 어떤 목록을 받아도 목록을 그대로 돌려줌"),
    (52, 52, "post", "<b>증명할 것</b>: 두 반복이 방문하는 원소가 같음"),
    (53, 55, "proof", "위 도우미를 불러 증명"),
    (57, 58, "body", "<b>반례 보조정리</b>: 조건이 없으면 실제로 다르다는 구체적인 예. 같음 보조정리는 \"조건이 있으면 같다\"만 말하므로, 이것이 있어야 \"조건이 꼭 필요하다\"고 말할 수 있음"),
    (59, 61, "post", "루프 body가 맨 앞 원소를 지우면, [1, 2, 3]에서 인덱스 반복은 1, 3을 방문하고(2를 건너뜀) for-each는 1, 2, 3을 방문"),
    (62, 64, "body", ""),
], start=20, end=64, title="RewriteRules.dfy: 규칙 ②")

**③ 변수 인라인: 조건 "루프 body가 식의 값을 바꾸지 않음"**. ②처럼 상태와 반복 루프 body를 아무것으로나 두고 일반적으로 적습니다.

In [10]:
lab.annotate("rewrite/RewriteRules.dfy", [
    (66, 72, "body", "규칙 ③. 상태는 무엇이든 됨(타입 <code>S</code>). 식 <code>e</code>는 상태의 함수 <code>size</code>, 반복 루프 body는 \"상태 → 루프 body가 끝난 뒤의 상태\" 함수 <code>body</code>로 나타냄. 두 반복이 <b>끝난 뒤의 상태</b>를 비교"),
    (73, 73, "body", ""),
    (74, 79, "func", "<b>바꾸기 전</b>: <code>n</code>을 반복 전에 <b>한 번</b> 계산해 두고 <code>n</code>번 돎(<code>int n = e;</code>)"),
    (81, 86, "func", "<b>바꾼 뒤</b>: 매번 <b>지금 상태</b>에서 <code>size</code>를 다시 계산해 비교(<code>i &lt; e</code>). <code>fuel</code>은 반복이 끝난다는 근거"),
    (88, 97, "proof", "증명 도우미: 루프 body가 <code>size</code>를 바꾸지 않으면 두 반복이 한 단계씩 같게 진행됨을 하나씩 늘려 가며 증명"),
    (99, 99, "body", "<b>같음 보조정리</b>"),
    (100, 100, "pre", "<b>조건</b>: 루프 body가 식의 값을 바꾸지 않음. 어떤 상태 <code>t</code>에서도 루프 body를 한 번 돈 뒤의 <code>size</code>가 그대로"),
    (101, 101, "post", "<b>증명할 것</b>: 처음 <code>size(s)</code>번 도는 반복과, 매번 <code>size</code>와 비교하는 반복이 끝난 뒤의 상태가 같음"),
    (102, 104, "proof", "위 도우미(<code>Loops</code>)를 불러 증명"),
    (106, 109, "body", "<b>반례 보조정리</b>: 조건이 없으면 실제로 다르다는 구체적인 예. 이것이 있어야 \"조건이 꼭 필요하다\"고 말할 수 있음"),
    (110, 113, "post", "상태는 목록, 식은 목록 길이, 루프 body는 맨 앞을 하나 뺌. [1, 2, 3]에서 변수를 쓰면 3번 돌아 <code>[]</code>, 인라인하면 [2, 3] → [3]에서 <code>2 &lt; 1</code>이 거짓이라 멈춰 <code>[3]</code>. E-2에서 반복이 한 번 일찍 끝난 것과 같은 일"),
    (114, 116, "body", ""),
], start=66, end=116, title="RewriteRules.dfy: 규칙 ③")

### 해 볼 것: 조건을 지우면 증명이 되지 않는지 보기

규칙 ③의 같음 보조정리에서 조건 줄(100줄 `requires forall t :: size(body(t)) == size(t)`)을 지웁니다. 줄 번호가 바뀌지 않도록 주석으로 바꿉니다.

In [11]:
F = "rewrite/RewriteRules.dfy"
lab.patch(F, "    requires forall t :: size(body(t)) == size(t)        // condition: the body does not change e\n",
             "    // requires ...   <- 지운 조건\n")

바꿨습니다: rewrite/RewriteRules.dfy   (되돌리기: lab.restore('rewrite/RewriteRules.dfy'))


In [12]:
lab.run("rules")

$ make rules
Dafny RewriteRules.dfy                         검증 14개, 오류 1개


**증명이 실패합니다.** 보고서는 이렇게 읽습니다.

- **빨간 줄 103줄**: 같음 보조정리가 증명 도우미 `Loops`를 부르는 곳. 이 호출의 사전조건을 보일 수 없습니다.
- **노란 줄 89줄**: `Loops`의 사전조건 "루프 body가 `size`를 바꾸지 않음"이 필요한데, 지운 조건이 없으니 이것을 알 수 없습니다.

조건이 없으면 "두 반복이 같다"를 증명할 수 없다는 뜻입니다. 실제로 조건이 없을 때 두 반복이 다르다는 것은 반례 보조정리(109~115줄)가 이미 증명해 두었습니다.

**꼭 되돌립니다.** 되돌리지 않으면 규칙 파일에 지운 조건이 남습니다.

In [13]:
lab.restore(F)
lab.run("rules")

되돌렸습니다: rewrite/RewriteRules.dfy
$ make rules
Dafny RewriteRules.dfy                         검증 15개, 오류 0개


## 조건을 확인하면서 적용하는 도구

조건부 다시쓰기 규칙을 표현하는 도구들이 있습니다. 조건을 어디까지 표현하는지가 다릅니다.

| 도구 | 조건을 거는 방식 |
|---|---|
| Spoon (Java) | 변환을 Java 코드로 쓰고, 조건도 코드로 검사. 필요한 분석은 직접 구현 |
| Error Prone + Refaster (Java) | 바꾸기 전·후 모양을 템플릿으로 쓰고, 패턴 변수에 판별기를 붙임. 검사기에서는 null 분석 같은 값 흐름 분석 결과를 쓸 수 있음 |
| Semgrep (여러 언어) | 패턴 + 제외 조건 + 자동 수정. 메서드 안의 값 흐름 추적 |
| IntelliJ 구조 검색·바꾸기 (Java) | 패턴 변수마다 타입, 정규식, 스크립트 조건 |
| Coccinelle (C) | 의미 패치에 실행 경로 조건(`when`)을 걺 |
| Rascal, Stratego, TXL | 조건부 다시쓰기 규칙과 적용 전략을 일반적으로 표현 |

변수 인라인의 조건("쓰는 동안 값이 안 바뀜")을 확인하려면, 반복문 안에서 `queue.size()`의 값을 바꾸는 호출(`poll`)이 있는지 알아야 합니다. 값이 어디서 바뀌는지 메서드를 넘어 따라가는 **값 흐름 분석**이 필요합니다. 도구가 조건을 표현할 수 있어도, 조건의 판단은 그 분석이 얼마나 정확한가에 달려 있습니다. 그래서 적용한 뒤 동작을 비교해 확인합니다.